# GPS Mapping Algorithm Test

Use this notebook to test `Locate via GPS` against a known hub and floor plan.

The notebook mirrors the current application algorithm in `collector/locate_via_gps.py`:

- Load `<building>/gps-map.json`.
- Read the selected floor's `referencePoints`.
- Derive an average longitude-to-`xPct` scale from all anchor pairs.
- Derive an average latitude-to-`yPct` scale from all anchor pairs.
- Use the first anchor as the reference point.
- Clamp the result to the map bounds.

For each test GPS point, it reports both percentage coordinates and absolute floor-plan `x, y` pixels, then overlays the predicted point on the floor plan for visual checking.

In [1]:
from __future__ import annotations

import base64
import json
import math
import re
import struct
import xml.etree.ElementTree as ET
from decimal import Decimal, InvalidOperation
from html import escape
from pathlib import Path
from typing import Any

from IPython.display import HTML, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "collector").exists() and (candidate / "assets").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing collector/ and assets/.")


REPO_ROOT = find_repo_root()
ASSETS_DIR = REPO_ROOT / "assets"
MANIFEST_PATH = ASSETS_DIR / "buildings.manifest.json"

print(f"Repo root: {REPO_ROOT}")
print(f"Assets dir: {ASSETS_DIR}")

Repo root: /mnt/c/Users/LiBayi/Documents/work/indoor-publicspace-collector
Assets dir: /mnt/c/Users/LiBayi/Documents/work/indoor-publicspace-collector/assets


## 1. Choose Hub, Floor, Latitude, And Longitude

Run the next cell and enter the GPS point you collected on site. Press Enter for `building_id` and `floor_id` to use the Bukit Canberra site plan defaults.

In [2]:
DEFAULT_BUILDING_ID = "Bukit-Canberra"
DEFAULT_FLOOR_ID = "Site_Plan"


def prompt_text(label: str, default: str) -> str:
    value = input(f"{label} [{default}]: ").strip()
    return value or default


def prompt_decimal(label: str, default: Decimal | None = None) -> Decimal:
    suffix = f" [{default}]" if default is not None else ""
    while True:
        raw_value = input(f"{label}{suffix}: ").strip()
        if not raw_value and default is not None:
            return default
        try:
            return Decimal(raw_value)
        except (InvalidOperation, ValueError):
            print("Enter a numeric latitude/longitude value.")


building_id = prompt_text("Building id", DEFAULT_BUILDING_ID)
floor_id = prompt_text("Floor id", DEFAULT_FLOOR_ID)

# Defaults use one existing Bukit Canberra anchor so the notebook can be run once before entering field data.
latitude = prompt_decimal("Latitude", Decimal("1.4479221215189448"))
longitude = prompt_decimal("Longitude", Decimal("103.8259331930616"))

print({"building_id": building_id, "floor_id": floor_id, "latitude": str(latitude), "longitude": str(longitude)})

{'building_id': 'First_Floor', 'floor_id': 'Site_Plan', 'latitude': '1.4479221215189448', 'longitude': '103.8259331930616'}


## 2. Current Application Algorithm

These helpers are a direct notebook translation of the current backend implementation. Keep this section aligned with `collector/locate_via_gps.py` when changing the production algorithm.

In [3]:
class GPSMappingError(Exception):
    pass


def to_decimal(value: Any, label: str) -> Decimal:
    try:
        return Decimal(str(value))
    except (InvalidOperation, ValueError, TypeError):
        raise GPSMappingError(f"Invalid {label} value.")


def parse_decimal_field(source: dict[str, Any], keys: tuple[str, ...], label: str) -> Decimal:
    for key in keys:
        if key not in source or source[key] is None:
            continue
        try:
            return Decimal(str(source[key]))
        except (InvalidOperation, ValueError):
            break
    raise GPSMappingError(f"GPS map anchor is missing a valid {label} value.")


def parse_anchor(data: dict[str, Any]) -> dict[str, Decimal]:
    return {
        "xPct": parse_decimal_field(data, ("xPct", "x", "xPercent"), "xPct"),
        "yPct": parse_decimal_field(data, ("yPct", "y", "yPercent"), "yPct"),
        "latitude": parse_decimal_field(data, ("latitude", "lat"), "latitude"),
        "longitude": parse_decimal_field(data, ("longitude", "lon", "lng"), "longitude"),
    }


def load_building_calibration(building_id: str) -> dict[str, Any]:
    gps_map_path = ASSETS_DIR / building_id / "gps-map.json"
    if not gps_map_path.exists():
        raise GPSMappingError(f"Missing GPS map: {gps_map_path}")
    with gps_map_path.open("r", encoding="utf-8") as file_obj:
        payload = json.load(file_obj)
    if not isinstance(payload, dict):
        raise GPSMappingError(f"GPS map must be a JSON object: {gps_map_path}")
    return payload


def get_floor_payload(building_data: dict[str, Any], floor_id: str) -> dict[str, Any]:
    if floor_id in building_data and isinstance(building_data[floor_id], dict):
        return building_data[floor_id]
    lowered = floor_id.lower()
    for key, value in building_data.items():
        if isinstance(key, str) and key.lower() == lowered and isinstance(value, dict):
            return value
    raise GPSMappingError(f"GPS calibration is unavailable for floor: {floor_id}")


def load_floor_anchors(building_id: str, floor_id: str) -> list[dict[str, Decimal]]:
    floor_payload = get_floor_payload(load_building_calibration(building_id), floor_id)
    raw_points = None
    for key in ("referencePoints", "anchors", "points"):
        candidate = floor_payload.get(key)
        if isinstance(candidate, list) and candidate:
            raw_points = candidate
            break
    if not raw_points:
        raise GPSMappingError("No GPS anchors defined for this floor.")
    anchors = [parse_anchor(item) for item in raw_points if isinstance(item, dict)]
    if len(anchors) < 2:
        raise GPSMappingError("GPS calibration requires at least two anchor points.")
    return anchors


def derive_scale(anchors: list[dict[str, Decimal]], source_key: str, target_key: str) -> Decimal | None:
    total = Decimal("0")
    count = 0
    for left_index, left in enumerate(anchors[:-1]):
        for right in anchors[left_index + 1:]:
            source_delta = right[source_key] - left[source_key]
            if source_delta == 0:
                continue
            target_delta = right[target_key] - left[target_key]
            total += target_delta / source_delta
            count += 1
    if count == 0:
        return None
    return total / Decimal(count)


def clamp_percent(value: Decimal) -> Decimal:
    if value < Decimal("0"):
        return Decimal("0")
    if value > Decimal("100"):
        return Decimal("100")
    return value


def locate_map_point_from_gps_existing_algorithm(
    building_id: str,
    floor_id: str,
    latitude: Decimal,
    longitude: Decimal,
) -> dict[str, Decimal]:
    anchors = load_floor_anchors(building_id, floor_id)
    lat_value = to_decimal(latitude, "latitude")
    lon_value = to_decimal(longitude, "longitude")

    lon_scale = derive_scale(anchors, "longitude", "xPct")
    if lon_scale is None:
        raise GPSMappingError("GPS calibration needs at least two anchors with distinct longitudes.")

    lat_scale = derive_scale(anchors, "latitude", "yPct")
    if lat_scale is None:
        raise GPSMappingError("GPS calibration needs at least two anchors with distinct latitudes.")

    reference = anchors[0]
    x_pct = reference["xPct"] + (lon_value - reference["longitude"]) * lon_scale
    y_pct = reference["yPct"] + (lat_value - reference["latitude"]) * lat_scale

    return {"xPct": clamp_percent(x_pct), "yPct": clamp_percent(y_pct)}

## 3. Resolve Floor Plan And Convert To Absolute Pixels

In [4]:
def load_manifest() -> dict[str, Any]:
    if not MANIFEST_PATH.exists():
        return {"buildings": {}}
    with MANIFEST_PATH.open("r", encoding="utf-8") as file_obj:
        payload = json.load(file_obj)
    return payload if isinstance(payload, dict) else {"buildings": {}}


def resolve_floor_plan_path(building_id: str, floor_id: str) -> Path:
    manifest = load_manifest()
    building = (manifest.get("buildings") or {}).get(building_id) or {}
    floors = building.get("floors") or {}
    floor_payload = floors.get(floor_id)
    if not floor_payload:
        lowered = floor_id.lower()
        for key, value in floors.items():
            if isinstance(key, str) and key.lower() == lowered:
                floor_payload = value
                break
    if isinstance(floor_payload, dict) and floor_payload.get("mapSrc"):
        candidate = ASSETS_DIR / str(floor_payload["mapSrc"])
        if candidate.exists():
            return candidate

    building_dir = ASSETS_DIR / building_id
    for extension in (".svg", ".png", ".jpg", ".jpeg", ".webp"):
        candidate = building_dir / f"{floor_id}{extension}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not resolve floor plan for {building_id}/{floor_id}")


def parse_svg_length(value: str | None) -> float | None:
    if not value:
        return None
    match = re.match(r"^\s*([0-9.]+)", value)
    return float(match.group(1)) if match else None


def svg_dimensions(path: Path) -> tuple[float, float]:
    root = ET.parse(path).getroot()
    view_box = root.attrib.get("viewBox") or root.attrib.get("viewbox")
    if view_box:
        parts = [float(part) for part in re.split(r"[\s,]+", view_box.strip()) if part]
        if len(parts) == 4 and parts[2] > 0 and parts[3] > 0:
            return parts[2], parts[3]
    width = parse_svg_length(root.attrib.get("width"))
    height = parse_svg_length(root.attrib.get("height"))
    if width and height:
        return width, height
    raise ValueError(f"SVG has no usable viewBox/width/height: {path}")


def png_dimensions(path: Path) -> tuple[int, int]:
    with path.open("rb") as file_obj:
        header = file_obj.read(24)
    if header[:8] != b"\x89PNG\r\n\x1a\n":
        raise ValueError(f"Not a PNG file: {path}")
    width, height = struct.unpack(">II", header[16:24])
    return width, height


def image_dimensions(path: Path) -> tuple[float, float]:
    suffix = path.suffix.lower()
    if suffix == ".svg":
        return svg_dimensions(path)
    if suffix == ".png":
        return png_dimensions(path)
    raise ValueError(f"Dimension reader is implemented for SVG/PNG only: {path.name}")


floor_plan_path = resolve_floor_plan_path(building_id, floor_id)
map_width, map_height = image_dimensions(floor_plan_path)
mapped_point = locate_map_point_from_gps_existing_algorithm(building_id, floor_id, latitude, longitude)
x_abs = (mapped_point["xPct"] / Decimal("100")) * Decimal(str(map_width))
y_abs = (mapped_point["yPct"] / Decimal("100")) * Decimal(str(map_height))

print(f"Floor plan: {floor_plan_path.relative_to(REPO_ROOT)}")
print(f"Floor plan size: {map_width:.2f} x {map_height:.2f}")
print(f"Mapped percentage: xPct={mapped_point['xPct']:.4f}, yPct={mapped_point['yPct']:.4f}")
print(f"Mapped absolute pixels: x={x_abs:.2f}, y={y_abs:.2f}")

FileNotFoundError: Could not resolve floor plan for First_Floor/Site_Plan

In [5]:
floor_id

'Site_Plan'

## 4. Visual Verification Plot

The red marker is the GPS point predicted by the current algorithm. Blue markers are calibration anchors from `gps-map.json`.

In [ ]:
def file_to_data_url(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".svg":
        mime = "image/svg+xml"
    elif suffix == ".png":
        mime = "image/png"
    else:
        mime = "application/octet-stream"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


def marker_html(point: dict[str, Any], *, color: str, label: str, size: int = 18) -> str:
    x_pct_value = float(point["xPct"])
    y_pct_value = float(point["yPct"])
    safe_label = escape(label)
    return f"""
      <div title=\"{safe_label}\" style=\"
        position:absolute;
        left:{x_pct_value:.6f}%;
        top:{y_pct_value:.6f}%;
        width:{size}px;
        height:{size}px;
        margin-left:{-size / 2}px;
        margin-top:{-size / 2}px;
        border:3px solid white;
        border-radius:50%;
        background:{color};
        box-shadow:0 1px 6px rgba(0,0,0,.45);
        z-index:3;
      "></div>
      <div style=\"
        position:absolute;
        left:{x_pct_value:.6f}%;
        top:{y_pct_value:.6f}%;
        transform:translate(10px, -18px);
        padding:2px 5px;
        background:rgba(255,255,255,.92);
        border:1px solid rgba(0,0,0,.25);
        border-radius:4px;
        font:12px system-ui, sans-serif;
        z-index:4;
        white-space:nowrap;
      ">{safe_label}</div>
    """


anchors = load_floor_anchors(building_id, floor_id)
markers = []
for index, anchor in enumerate(anchors, start=1):
    markers.append(marker_html(anchor, color="#2563eb", label=f"A{index}", size=14))
markers.append(marker_html(mapped_point, color="#dc2626", label="GPS", size=20))

image_url = file_to_data_url(floor_plan_path)
display(HTML(f"""
<div style="font:14px system-ui, sans-serif; margin:8px 0 14px;">
  <div><strong>{escape(building_id)} / {escape(floor_id)}</strong></div>
  <div>GPS input: lat {escape(str(latitude))}, lng {escape(str(longitude))}</div>
  <div>Mapped: xPct {float(mapped_point['xPct']):.4f}, yPct {float(mapped_point['yPct']):.4f}, x {float(x_abs):.2f}px, y {float(y_abs):.2f}px</div>
</div>
<div style="position:relative; display:inline-block; width:min(100%, 1000px); border:1px solid #d1d5db; background:#fff;">
  <img src="{image_url}" style="display:block; width:100%; height:auto;" />
  {''.join(markers)}
</div>
"""))

## 5. Anchor Diagnostics

This table feeds each calibration anchor back into the current algorithm. Large residuals mean the current algorithm cannot reproduce the calibration points well. That usually means either the anchors need correction, or the floor plan is rotated/skewed enough that the algorithm needs a richer transform than separate `longitude -> x` and `latitude -> y` scales.

In [ ]:
def anchor_residual_rows() -> list[dict[str, Any]]:
    rows = []
    for index, anchor in enumerate(anchors, start=1):
        predicted = locate_map_point_from_gps_existing_algorithm(
            building_id,
            floor_id,
            anchor["latitude"],
            anchor["longitude"],
        )
        dx_pct = predicted["xPct"] - anchor["xPct"]
        dy_pct = predicted["yPct"] - anchor["yPct"]
        dx_px = (dx_pct / Decimal("100")) * Decimal(str(map_width))
        dy_px = (dy_pct / Decimal("100")) * Decimal(str(map_height))
        pixel_error = Decimal(str(math.hypot(float(dx_px), float(dy_px))))
        rows.append({
            "anchor": f"A{index}",
            "anchor_xPct": anchor["xPct"],
            "anchor_yPct": anchor["yPct"],
            "pred_xPct": predicted["xPct"],
            "pred_yPct": predicted["yPct"],
            "dx_pct": dx_pct,
            "dy_pct": dy_pct,
            "dx_px": dx_px,
            "dy_px": dy_px,
            "pixel_error": pixel_error,
        })
    return rows


def print_rows(rows: list[dict[str, Any]]) -> None:
    headers = ["anchor", "anchor_xPct", "anchor_yPct", "pred_xPct", "pred_yPct", "dx_pct", "dy_pct", "dx_px", "dy_px", "pixel_error"]
    print(" | ".join(headers))
    print(" | ".join(["---"] * len(headers)))
    for row in rows:
        values = []
        for header in headers:
            value = row[header]
            if isinstance(value, Decimal):
                values.append(f"{value:.4f}")
            else:
                values.append(str(value))
        print(" | ".join(values))


residual_rows = anchor_residual_rows()
print_rows(residual_rows)

## 6. Calibration Helper

If the red marker is wrong, estimate the correct absolute `x, y` pixel position on the floor plan and run this cell. It prints the error and a JSON anchor candidate you can copy into `gps-map.json` after checking it.

Leave either field blank to skip.

In [ ]:
expected_x_text = input("Expected x pixel on floor plan, blank to skip: ").strip()
expected_y_text = input("Expected y pixel on floor plan, blank to skip: ").strip()

if expected_x_text and expected_y_text:
    expected_x = Decimal(expected_x_text)
    expected_y = Decimal(expected_y_text)
    expected_x_pct = (expected_x / Decimal(str(map_width))) * Decimal("100")
    expected_y_pct = (expected_y / Decimal(str(map_height))) * Decimal("100")
    dx = x_abs - expected_x
    dy = y_abs - expected_y
    error_px = math.hypot(float(dx), float(dy))

    print(f"Expected percentage: xPct={expected_x_pct:.4f}, yPct={expected_y_pct:.4f}")
    print(f"Prediction error: dx={dx:.2f}px, dy={dy:.2f}px, total={error_px:.2f}px")
    print("Suggested anchor candidate for gps-map.json:")
    print(json.dumps({
        "xPct": round(float(expected_x_pct), 4),
        "yPct": round(float(expected_y_pct), 4),
        "latitude": float(latitude),
        "longitude": float(longitude),
    }, indent=2))
else:
    print("Skipped calibration helper.")